05/12-2025

> **Context:** Be sure to check the `README`, in this folder, for a proper overview before diving in.

This notebook serves as a documentation archive for the scripts in this folder. It is intended for reading, **not execution**.
(For that, we recommend using the Python scripts and the job scripts). 

#### Data Preprocessing Note
This model handles data scaling differently than the others:
1.  **Scaling:** Both Gene (X) and Transcript (Y) data are normalized using `StandardScaler`.
2.  **Evaluation:** To maintain consistency with other models (which only scale X), we apply an **inverse transform** to the outputs before calculating the MSE. This ensures all performance metrics are reported in the same unit space.

### **The code of `pca_script.py`:**

The first PCA-based FFNN script that was created. Its main purpose was that of an mostly exploratory script. 

In [ ]:
import os
import json
import time
import csv
import torch
import numpy as np
import matplotlib
# [CRITICAL] Force matplotlib to use non-interactive backend for HPC
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------
# Config
# ------------------------------
SEED            = 42
TEST_FRAC       = 0.15
VAL_FRAC        = 0.15
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

PCA_DIM         = 1000  # Number of PCA components
BATCH_SIZE      = 128
N_EPOCHS        = 30
LR              = 1e-3

# ------------------------------
# Dynamic Output Paths
# ------------------------------
# Try to get the LSF Job ID. If running locally, use a timestamp.
JOB_ID = os.environ.get("LSB_JOBID")
if JOB_ID is None:
    JOB_ID = f"local_{time.strftime('%Y%m%d_%H%M%S')}"

print(f"[INFO] Detected Job ID: {JOB_ID}")

OUT_DIR         = "PCA/results"
# Append Job ID to filenames so they don't overwrite each other
FIG_PATH        = os.path.join(OUT_DIR, f"pca_training_curve_{JOB_ID}.png")
JSON_PATH       = os.path.join(OUT_DIR, f"pca_metrics_{JOB_ID}.json")
CSV_PATH        = os.path.join(OUT_DIR, f"pca_training_log_{JOB_ID}.csv")

# Ensure output directory exists
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------
# Seed | Ensuring reproducibility
# ------------------------------
torch.manual_seed(SEED)
np.random.seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ------------------------------
# Data Loading
# ------------------------------
DATA_PT = os.environ.get("DATA_PT")
if DATA_PT is None:
    bh = os.environ.get("BLACKHOLE")
    user = os.environ.get("USER")
    if bh is None or user is None:
        DATA_PT = "data.pt" 
    else:
        DATA_PT = os.path.join(bh, user, "data.pt")

print(f"[INFO] Loading tensors from: {DATA_PT}")
data = torch.load(DATA_PT, map_location="cpu", weights_only=False)

X = data["Xg_log1p"].float().cpu().numpy()
Y = torch.log1p(data["Y_tx"].float()).cpu().numpy()

# ------------------------------
# Data Processing
# ------------------------------
print("[INFO] Splitting, Scaling, and running PCA...")

# 1. Split
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=TEST_FRAC, random_state=SEED, shuffle=True
)
relative_val_size = VAL_FRAC / (1 - TEST_FRAC)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_temp, Y_temp, test_size=relative_val_size, random_state=SEED, shuffle=True
)

# 2. Standardize
scaler_x = StandardScaler()
X_train = scaler_x.fit_transform(X_train)
X_val   = scaler_x.transform(X_val)
X_test  = scaler_x.transform(X_test)

scaler_y = StandardScaler()
Y_train = scaler_y.fit_transform(Y_train)
Y_val   = scaler_y.transform(Y_val)
Y_test  = scaler_y.transform(Y_test)

# 3. PCA
pca = PCA(n_components=PCA_DIM)
X_train_pca = pca.fit_transform(X_train)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)

# 4. To Tensor
X_train_t = torch.tensor(X_train_pca, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_pca,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_pca,  dtype=torch.float32)

Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
Y_val_t   = torch.tensor(Y_val,   dtype=torch.float32)
Y_test_t  = torch.tensor(Y_test,  dtype=torch.float32)

# 5. DataLoaders
train_dataset = TensorDataset(X_train_t, Y_train_t)
val_dataset   = TensorDataset(X_val_t,   Y_val_t)
test_dataset  = TensorDataset(X_test_t,  Y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"[INFO] Training batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ------------------------------
# Model
# ------------------------------
class IsoformPredictor(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------
# Training Setup
# ------------------------------
input_dim = X_train_t.shape[1]
output_dim = Y_train_t.shape[1]

model = IsoformPredictor(input_dim, output_dim).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss() 

# ------------------------------
# Training Loop
# ------------------------------
print(f"[INFO] Starting training on {DEVICE}...")
train_losses = []
val_losses = []
start_time = time.time()

for epoch in range(N_EPOCHS):
    model.train()
    total_train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1:02d}/{N_EPOCHS}] | Train Loss (Scaled): {avg_train_loss:.4f} | Val Loss (Scaled): {avg_val_loss:.4f}")

total_time = time.time() - start_time

# ------------------------------
# Final Test Evaluation (unscaled data)
# ------------------------------

#Note: In order to make the model comparable to the non-PCA script, we need to inverse transform
#the predictions back to the original log1p scale before calculating the final MSE! 

print("\n[INFO] Running Final Test Evaluation (Unscaling back to log1p)...")
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        # Get Scaled Predictions
        preds_scaled = model(xb)
        
        # Move to CPU and store
        all_preds.append(preds_scaled.cpu().numpy())
        all_targets.append(yb.numpy()) # yb was already on CPU in DataLoader, or move .cpu() if needed

# 1. Concatenate all batches
P_scaled = np.vstack(all_preds)
T_scaled = np.vstack(all_targets)

# 2. Inverse Transform (Back to original log1p scale)
# This makes it comparable to the other non-scaled models
P_log1p = scaler_y.inverse_transform(P_scaled)
T_log1p = scaler_y.inverse_transform(T_scaled)

# 3. Calculate MSE on the unscaled data
final_test_mse_unscaled = ((P_log1p - T_log1p)**2).mean()

print(f"[FINAL] Test MSE (Scaled):   {((P_scaled - T_scaled)**2).mean():.4f}")
print(f"[FINAL] Test MSE (Unscaled) - Comparable version: {final_test_mse_unscaled:.4f}")

# ------------------------------
# Save Results & Plots
# ------------------------------

# 1. Save Training Curve Plot
try:
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss (Scaled)', marker='o')
    plt.plot(val_losses, label='Validation Loss (Scaled)', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss (Scaled)')
    plt.title(f'PCA FFNN (PCA={PCA_DIM}) | Job: {JOB_ID}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(FIG_PATH, dpi=150)
    plt.close()
    print(f"[INFO] Plot saved to: {FIG_PATH}")
except Exception as e:
    print(f"[WARN] Failed to save plot: {e}")

# 2. Save Metrics to JSON
summary = {
    "meta": {
        "job_id": JOB_ID,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "config": {
        "pca_dim": PCA_DIM,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "epochs": N_EPOCHS,
        "seed": SEED
    },
    "results": {
        "final_train_mse_scaled": train_losses[-1],
        "final_val_mse_scaled": val_losses[-1],
        "test_mse_scaled": float(((P_scaled - T_scaled)**2).mean()),
        "test_mse_unscaled": float(final_test_mse_unscaled), # This is the key metric for comparing our models. The scaled MSE is less relevant. 
        "total_time_sec": round(total_time, 2)
    }
}
with open(JSON_PATH, "w") as f:
    json.dump(summary, f, indent=4)
print(f"[INFO] Metrics saved to: {JSON_PATH}")

# 3. Save Logs to CSV
with open(CSV_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_mse_scaled", "val_mse_scaled"])
    for i in range(N_EPOCHS):
        writer.writerow([i+1, train_losses[i], val_losses[i]])

print("[INFO] All done.")


### **The code of `pca_dim_search.py`:**

The script which explores multiple (user-defined) different PCA dimensions in order determine the one which leads to best performance. <br>
<br>
Admittedly, the algorithm would arguably more proper in exploring PCA dimensions, if it hadn't kept the hyperparameters fixed. <br>
Although that would also have severely increase the computational resource and time demand. <br>
\- which we decided wasn't a justifiable use of the HPC cluster's resources. <br>
<br>
Therefore we went for this more resource friendly approach. 

In [ ]:
#!/usr/bin/env python3

#The purpose of this script is similar to multiple_pca_script.py, 
#but where we test out 5 different values for the number of principal components to retain in PCA.
#I.e. how many dimensions we reduce to. 


"""
PCA Dimension Sweep (GPU + AMP)
Tests how model performance changes with different input dimensions (PCA components).
Keeps model hyperparameters FIXED.

Output: Metrics unscaled back to log1p space.
"""

import os
import json
import time
import csv
import random
import pathlib
import math

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

import matplotlib
# Safe backend for HPC (No screen)
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------
# Config
# ------------------------------
SEED      = 42
TEST_FRAC = 0.15
VAL_FRAC  = 0.15
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
AMP       = (DEVICE == "cuda")

# ---- EXPERIMENT SETUP ----
#These are the PCA dimensions (number of PCA components) which will be tested:
PCA_DIMS_TO_TEST = [100, 500, 1000, 2000, 3000, 5000, 10000]

#The script will try to find the PCA components which give the best performance. 


#Fixed Hyperparameters - the only thing which will change is the PCA input dimensions.
#These fixed hyperparameters were chosen based on prior experimentation and a small mix of arbitrary choices. 

FIXED_HP = {
    "hidden": [1024, 512],
    "act": "gelu",
    "lr": 1e-3,
    "batch_size": 128,
    "dropout": 0.1,
    "batchnorm": True,
    "epochs": 30
}

GRAD_CLIP  = 1.0
EVAL_EVERY = 5

# ------------------------------
# Output Paths
# ------------------------------
JOB_ID = os.environ.get("LSB_JOBID")
if JOB_ID is None:
    JOB_ID = f"local_{time.strftime('%Y%m%d_%H%M%S')}"

print(f"[INFO] Detected Job ID: {JOB_ID}")

BASE_DIR = "PCA/results"
os.makedirs(BASE_DIR, exist_ok=True)
TRIAL_FIG_DIR = os.path.join(BASE_DIR, f"figs_pca_sweep_{JOB_ID}")

RESULTS_CSV  = os.path.join(BASE_DIR, f"pca_sweep_results_{JOB_ID}.csv")
SUMMARY_JSON = os.path.join(BASE_DIR, f"pca_sweep_summary_{JOB_ID}.json")
SWEEP_PLOT   = os.path.join(BASE_DIR, f"pca_sweep_plot_{JOB_ID}.png")

# ------------------------------
# Reproducibility
# ------------------------------
def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)

# ------------------------------
# Data Loading & PCA Processing
# ------------------------------
DATA_PT = os.environ.get("DATA_PT")
if DATA_PT is None:
    bh = os.environ.get("BLACKHOLE")
    user = os.environ.get("USER")
    if bh is None or user is None:
        DATA_PT = "data.pt"
    else:
        DATA_PT = os.path.join(bh, user, "data.pt")

print(f"[INFO] Loading tensors from: {DATA_PT}")
data = torch.load(DATA_PT, map_location="cpu", weights_only=False)

X = data["Xg_log1p"].float().cpu().numpy()
Y = torch.log1p(data["Y_tx"].float()).cpu().numpy()

print("[INFO] Processing Data...")

# 1. Split
X_temp, X_test, Y_temp, Y_test = train_test_split(X, Y, test_size=TEST_FRAC, random_state=SEED, shuffle=True)
rel_val = VAL_FRAC / (1 - TEST_FRAC)
X_train, X_val, Y_train, Y_val = train_test_split(X_temp, Y_temp, test_size=rel_val, random_state=SEED, shuffle=True)

# 2. Scale
scaler_x = StandardScaler().fit(X_train)
X_train = scaler_x.transform(X_train)
X_val   = scaler_x.transform(X_val)
X_test  = scaler_x.transform(X_test)

scaler_y = StandardScaler().fit(Y_train)
Y_train = scaler_y.transform(Y_train)
Y_val   = scaler_y.transform(Y_val)
Y_test  = scaler_y.transform(Y_test)

# 3. PCA (Using the highest PCA dimension to fit once)
max_dim = max(PCA_DIMS_TO_TEST)
print(f"[INFO] Fitting PCA with n_components={max_dim} (Max required)...")
pca = PCA(n_components=max_dim).fit(X_train) #We only wanna calculate the PCA once. Therefore we are using the max dimensions. 

X_train_pca = pca.transform(X_train)
X_val_pca   = pca.transform(X_val)
X_test_pca  = pca.transform(X_test)

# 4. To Tensor (again, remember, we are using the max dimensions here, as there is no point in recalculating PCA multiple times)
Xt_train_full = torch.tensor(X_train_pca, dtype=torch.float32)
Xt_val_full   = torch.tensor(X_val_pca,   dtype=torch.float32)
Xt_test_full  = torch.tensor(X_test_pca,  dtype=torch.float32)

Yt_train = torch.tensor(Y_train, dtype=torch.float32)
Yt_val   = torch.tensor(Y_val,   dtype=torch.float32)
Yt_test  = torch.tensor(Y_test,  dtype=torch.float32)

# ------------------------------
# Model Definition
# ------------------------------

#Defining activation function and FFNN class - the model will be a bit further defined in the train_and_evaluate function:

def get_activation(name: str):
    if name == "relu": return nn.ReLU()
    if name == "gelu": return nn.GELU()
    if name == "tanh": return nn.Tanh()
    if "leaky" in name: return nn.LeakyReLU(0.01)
    return nn.ReLU()

class FFNN(nn.Module):
    def __init__(self, in_dim, out_dim, hp):
        super().__init__()
        layers, prev = [], in_dim
        for h in hp["hidden"]:
            layers.append(nn.Linear(prev, h))
            if hp["batchnorm"]: layers.append(nn.BatchNorm1d(h))
            layers.append(get_activation(hp["act"]))
            if hp["dropout"] > 0: layers.append(nn.Dropout(hp["dropout"]))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

##########################################

# ------------------------------
# Training Function
# ------------------------------
def train_and_evaluate(current_pca_dim):
    print(f"\n>>> Starting Run: PCA_DIM = {current_pca_dim}")
    
    # Slice Tensors for current dimension
    xt_tr = Xt_train_full[:, :current_pca_dim]
    xt_va = Xt_val_full[:,   :current_pca_dim]
    xt_te = Xt_test_full[:,  :current_pca_dim]
    
    # Loaders
    tr_loader = DataLoader(TensorDataset(xt_tr, Yt_train), batch_size=FIXED_HP["batch_size"], shuffle=True, drop_last=True)
    va_loader = DataLoader(TensorDataset(xt_va, Yt_val),   batch_size=FIXED_HP["batch_size"], shuffle=False)
    te_loader = DataLoader(TensorDataset(xt_te, Yt_test),  batch_size=FIXED_HP["batch_size"], shuffle=False)

    #Model - defines the model, optimizer, scheduler, criterion, scaler. 
    model = FFNN(current_pca_dim, Y.shape[1], FIXED_HP).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=FIXED_HP["lr"], weight_decay=1e-4)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    criterion = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    #Loop for training: 
    train_curve, val_curve = [], []
    t0 = time.time()
    
    for epoch in range(1, FIXED_HP["epochs"] + 1):
        model.train()
        loss_sum, counts = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                preds = model(xb)
                loss = criterion(preds, yb)
            scaler.scale(loss).backward()
            if GRAD_CLIP:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.item() * yb.size(0)
            counts += yb.size(0)
        train_curve.append(loss_sum/counts)

        #Validation
        if epoch == 1 or epoch % EVAL_EVERY == 0 or epoch == FIXED_HP["epochs"]:
            model.eval()
            vloss_sum, vcounts = 0.0, 0
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    p = model(xb)
                    vloss_sum += criterion(p, yb).item() * yb.size(0)
                    vcounts += yb.size(0)
            val_loss = vloss_sum/vcounts
            val_curve.append(val_loss)
            sch.step(val_loss)
            print(f"[PCA {current_pca_dim}] ep {epoch:02d} | val_scaled {val_loss:.4f}")

    #Final Test (on unscaled data)
    model.eval()
    preds_list, targs_list = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            xb = xb.to(DEVICE)
            p = model(xb)
            preds_list.append(p.cpu().numpy())
            targs_list.append(yb.numpy())
    
    P_scaled = np.vstack(preds_list)
    T_scaled = np.vstack(targs_list)
    
    #Inverse Transform to log1p space (Back to original scale)
    P_log1p = scaler_y.inverse_transform(P_scaled)
    T_log1p = scaler_y.inverse_transform(T_scaled)
    
    final_mse = ((P_log1p - T_log1p)**2).mean()
    
    #Save training curve for this specific PCA dimension: 
    pathlib.Path(TRIAL_FIG_DIR).mkdir(parents=True, exist_ok=True)
    plt.figure()
    plt.plot(train_curve, label="Train")
    plt.plot(np.linspace(1, FIXED_HP["epochs"], len(val_curve)), val_curve, "o-", label="Val")
    plt.title(f"PCA Dim: {current_pca_dim}")
    plt.legend()
    plt.savefig(os.path.join(TRIAL_FIG_DIR, f"curve_pca_{current_pca_dim}.png"))
    plt.close()

    #Explicitly cast to standard python float() for JSON safety. Otherwise numpy float types will cause an error!!
    return {
        "pca_dim": int(current_pca_dim), #Cast to int
        "test_mse_unscaled": float(final_mse), #Cast to float
        "test_mse_scaled": float(((P_scaled - T_scaled)**2).mean()), #Cast to float
        "time_sec": round(time.time() - t0, 1)
    }

#############################################

# ------------------------------
# Main Sweep Loop - test out the PCA dimensions that were inputted in the config section
# ------------------------------
results = []

# CSV Init
with open(RESULTS_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["pca_dim", "test_mse_unscaled", "test_mse_scaled", "time_sec"])
    w.writeheader()

print("\n=== STARTING PCA SWEEP ===")
for dim in PCA_DIMS_TO_TEST:
    res = train_and_evaluate(dim)
    results.append(res)
    
    # Update CSV
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=res.keys())
        w.writerow(res)
    
    print(f"[RESULT] Dim {dim} -> Test MSE (Unscaled): {res['test_mse_unscaled']:.4f}")
 
# ------------------------------
# Summary Plot
# ------------------------------
dims = [r["pca_dim"] for r in results]
mses = [r["test_mse_unscaled"] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(dims, mses, marker='o', linestyle='-', linewidth=2, color='b')
plt.xlabel("Number of PCA Components")
plt.ylabel("Test MSE (Unscaled)")
plt.title("Model Performance vs. PCA Dimension")
plt.grid(True, alpha=0.3)
plt.savefig(SWEEP_PLOT, dpi=150)
plt.close()

# Save JSON
with open(SUMMARY_JSON, "w") as f:
    json.dump({"fixed_hp": FIXED_HP, "results": results}, f, indent=4)

print(f"\n[INFO] Sweep Complete. Summary plot saved to {SWEEP_PLOT}")


### **The code of `multiple_pca_script.py`:**

This script was adapted from `ffnn_search/ffnn_search.py`. <br>
Its purpose is to determine the best configuration of hyperparameters for the PCA-based FFNN model. <br>
It uses a heuristic approach to pick hyperparameters to chose from. <br>
<br>
Just like `ffnn_search/ffnn_search.py`, this code has Stage 1 and a Stage 2 functionality, which the user manually needs to switch on. <br>
The stage 2 was never tried out, as this feature seemed more relevant for the raw data FFNN than the PCA-based FFNN. 

In [ ]:
#!/usr/bin/env python3

#03/12-2025


"""
PCA FFNN Random Search (GPU + AMP) — 2-stage
Adapted from ffnn_search_v2.py to use PCA-processed data.

Stage 1: Random search (100 trials, see N_TRIALS)
Stage 2: Retrain top-5 configs (see TOP_K_STAGE2)
Output: Metrics unscaled back to log1p space for direct comparison with raw models.
"""

import os, json, math, random, csv, time, pathlib, ast
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg') #This is done to make it a bit more safe when running it on the HPC
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------
# Global config
# ------------------------------
SEED      = 42
TEST_FRAC = 0.15
VAL_FRAC  = 0.15
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
AMP       = (DEVICE == "cuda")

#PCA components - we know from prior experiments that 1000 is a good value! 
PCA_DIM   = 1000

# ---- STAGE CONTROL ----
# STAGE = 1 | random search with pruning
# STAGE = 2 | retrain top-K configs from Stage 1 (no global pruning)

STAGE        = 1   #<--------------------- (!) UPDATE THIS VALUE TO 2, FOR STAGE 2 RUN (!)
# 1 = Search, 2 = Retrain Top-K

JOB_ID_STAGE1 = "INSERT_JOB_ID_FROM_STAGE1_HERE"  #<--- (!) UPDATE THIS VALUE FOR STAGE 2 RUN (!)

TOP_K_STAGE2 = 5          

GRAD_CLIP    = 1.0
EVAL_EVERY   = 5          

#Hyperparameter optimization, search ranges (same as ffnn_search_v2.py): 
BATCHES    = [128, 192, 256]
LRS        = [1e-3, 2e-3, 3e-3]
DROPOUTS   = [0.0, 0.1, 0.2]
BATCHNORMS = [False, True]
ACTS       = ["tanh", "relu", "gelu", "leakyrelu"]
DEPTH_CHOICES = [1, 2, 3]
WIDTH_CHOICES = [256, 512, 1024, 2048]

if STAGE == 1:
    N_TRIALS        = 100
    MAX_EPOCHS      = 50
    PATIENCE        = 5
    MIN_PRUNE_EPOCH = 30
    PRUNE_FACTOR    = 1.5
elif STAGE == 2:
    N_TRIALS        = TOP_K_STAGE2
    MAX_EPOCHS      = 200
    PATIENCE        = 10
    MIN_PRUNE_EPOCH = None
    PRUNE_FACTOR    = None

# ------------------------------
# Dynamic Output Paths
# ------------------------------

#Use Job ID to prevent overwriting results if you run multiple jobs
JOB_ID = os.environ.get("LSB_JOBID")
if JOB_ID is None:
    JOB_ID = f"local_{time.strftime('%Y%m%d_%H%M%S')}"

print(f"[INFO] Detected Job ID: {JOB_ID}")

BASE_DIR = "PCA/results"
os.makedirs(BASE_DIR, exist_ok=True)
TRIAL_FIG_DIR   = os.path.join(BASE_DIR, f"figs_trials_{JOB_ID}")

#Define paths based on stage (1 or 2) + job ID:
if STAGE == 1:
    RESULTS_CSV      = os.path.join(BASE_DIR, f"pca_results_stage1_{JOB_ID}.csv")
    SUMMARY_JSON     = os.path.join(BASE_DIR, f"pca_summary_stage1_{JOB_ID}.json")
    BEST_MODEL_PT    = os.path.join(BASE_DIR, f"pca_best_model_stage1_{JOB_ID}.pt")
    SUMMARY_FIG_BAR  = os.path.join(BASE_DIR, f"pca_bar_stage1_{JOB_ID}.png")
    SUMMARY_FIG_TOP5 = os.path.join(BASE_DIR, f"pca_top5_stage1_{JOB_ID}.png")
elif STAGE == 2:
    RESULTS_CSV      = os.path.join(BASE_DIR, f"pca_results_stage2_{JOB_ID}.csv")
    SUMMARY_JSON     = os.path.join(BASE_DIR, f"pca_summary_stage2_{JOB_ID}.json")
    BEST_MODEL_PT    = os.path.join(BASE_DIR, f"pca_best_model_stage2_{JOB_ID}.pt")
    SUMMARY_FIG_BAR  = os.path.join(BASE_DIR, f"pca_bar_stage2_{JOB_ID}.png")
    SUMMARY_FIG_TOP5 = os.path.join(BASE_DIR, f"pca_top5_stage2_{JOB_ID}.png")

# IMPORTANT: To run Stage 2, you must point this to the SPECIFIC CSV file from Stage 1

#Update this filename manually AFTER Stage 1 finishes!
STAGE1_INPUT_CSV = f"PCA/results/pca_results_stage1_{JOB_ID_STAGE1}.csv" #Remember to update for stage 2, in the config section (!)

# ------------------------------
# Reproducibility and setup
# ------------------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)

# ------------------------------
# Data Loading & PCA Processing
# ------------------------------
DATA_PT = os.environ.get("DATA_PT")
if DATA_PT is None:
    bh = os.environ.get("BLACKHOLE")
    user = os.environ.get("USER")
    if bh is None or user is None:
        DATA_PT = "data.pt"
    else:
        DATA_PT = os.path.join(bh, user, "data.pt")

print(f"[INFO] Loading tensors from: {DATA_PT}")
data = torch.load(DATA_PT, map_location="cpu", weights_only=False)

X = data["Xg_log1p"].float().cpu().numpy()
Y = torch.log1p(data["Y_tx"].float()).cpu().numpy()

########################################################################

#Data processing: Split -> Scale -> PCA -> To Tensor -> DataLoaders

print("[INFO] Processing Data (Split -> Scale -> PCA)...")

# 1. Split
X_temp, X_test, Y_temp, Y_test = train_test_split(X, Y, test_size=TEST_FRAC, random_state=SEED, shuffle=True)
rel_val = VAL_FRAC / (1 - TEST_FRAC)
X_train, X_val, Y_train, Y_val = train_test_split(X_temp, Y_temp, test_size=rel_val, random_state=SEED, shuffle=True)

# 2. Scale (StandardScaler on X AND Y)
# We scale Y to help the neural network train better, but we will unscale later for comparison
scaler_x = StandardScaler().fit(X_train)
X_train = scaler_x.transform(X_train)
X_val   = scaler_x.transform(X_val)
X_test  = scaler_x.transform(X_test)

scaler_y = StandardScaler().fit(Y_train)
Y_train = scaler_y.transform(Y_train)
Y_val   = scaler_y.transform(Y_val)
Y_test  = scaler_y.transform(Y_test)

# 3. PCA
pca = PCA(n_components=PCA_DIM).fit(X_train)
X_train = pca.transform(X_train)
X_val   = pca.transform(X_val)
X_test  = pca.transform(X_test)

print(f"[INFO] Data ready. Input dim: {X_train.shape[1]}")

# 4. To Tensor (Keep on CPU, move batches to GPU)
Xt_train = torch.tensor(X_train, dtype=torch.float32)
Yt_train = torch.tensor(Y_train, dtype=torch.float32)
Xt_val   = torch.tensor(X_val,   dtype=torch.float32)
Yt_val   = torch.tensor(Y_val,   dtype=torch.float32)
Xt_test  = torch.tensor(X_test,  dtype=torch.float32)
Yt_test  = torch.tensor(Y_test,  dtype=torch.float32)

# 5. DataLoaders
train_ds = TensorDataset(Xt_train, Yt_train)
val_ds   = TensorDataset(Xt_val, Yt_val)
test_ds  = TensorDataset(Xt_test, Yt_test)

def get_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=(shuffle and len(ds)>bs))

#########################################################################################

#Defining the model: 

# ------------------------------
# Model
# ------------------------------
def get_activation(name: str):
    n = name.lower()
    if n == "relu": return nn.ReLU()
    if n == "gelu": return nn.GELU()
    if n in ("leakyrelu", "leaky_relu"): return nn.LeakyReLU(0.01)
    if n == "tanh": return nn.Tanh()
    raise ValueError(f"Unknown activation: {name}")

class FFNN(nn.Module):
    def __init__(self, in_dim, out_dim, hidden, act="gelu", dropout=0.0, batchnorm=False):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            if batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(get_activation(act))
            if dropout > 0: layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ------------------------------
# Hyperparameter optimization helpers
# ------------------------------
def sample_arch(rnd):
    depth = rnd.choice(DEPTH_CHOICES) #Randomly choose depth of model (amount of layers). 
    hidden = [rnd.choice(WIDTH_CHOICES) for _ in range(depth)] #Amount of neurons in each hidden layer, randomly chosen. 
    return hidden

def sample_hp(t):
    rnd = random.Random(SEED + 1000 + t)
    hidden = sample_arch(rnd)
    name = f"L{len(hidden)}_" + "_".join(map(str, hidden))
    return {
        "name": f"{name}_t{t}",
        "hidden": hidden,
        "act": rnd.choice(ACTS),
        "dropout": rnd.choice(DROPOUTS),
        "batchnorm": rnd.choice(BATCHNORMS),
        "lr": rnd.choice(LRS),
        "batch_size": rnd.choice(BATCHES),
        "epochs": MAX_EPOCHS,
    }

####################################################################

#Training the model: 

# ------------------------------
# Train Loop
# ------------------------------
criterion = nn.MSELoss()

def train_once(hp, trial_seed, global_best_val=None, stage=1):
    set_seed(trial_seed)
    
    #Loaders specific to this batch size
    tr_loader = get_loader(train_ds, hp["batch_size"], True)
    va_loader = get_loader(val_ds, hp["batch_size"], False)

    model = FFNN(PCA_DIM, Y.shape[1], hp["hidden"], hp["act"], hp["dropout"], hp["batchnorm"]).to(DEVICE)
    
    opt = optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=1e-4)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    best_val = math.inf
    best_state = None
    noimp = 0
    train_curve, val_curve = [], []

    t0 = time.time()
    for epoch in range(1, hp["epochs"] + 1):
        model.train()
        total_loss, counts = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                preds = model(xb)
                loss = criterion(preds, yb)
            scaler.scale(loss).backward()

            #Gradient Clipping - Dealing with exploding gradients: 
            if GRAD_CLIP: #Mostly as a safeguard, as some of the models can be quite large or otherwise act in unexpected ways. 
                
                #AMP and Gradient clipping background (as these 2 things were not really explained in the course): 
                
                #In order to avoid underflow (i.e. the limit for how small float numbers can be represented on a computer), you use AMPs.
                #Underflow e.g. cause very small gradient values to become zero. This creates a problem. 
                
                #AMP (Automatic Mixed Precision) is a manager which fixes the underflow problem by multipling the gradients 
                #by a scaling factor before backpropagation, in order to instead make them a really large number. 

                #Gradient clipping is a technique which helps fix exploding gradients by limiting the maximum size of the gradients. 
                #If gradients are larger than the limit (e.g. 1.0), then it shrinks all gradients proportionally 
                #so that all the gradients fit under the clipping limit. 

                scaler.unscale_(opt) #Undo the AMP scaling.
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP) #Clip the gradients, so they are below the gradient clipping limit. 
                #Do this not to get exploding gradients. 

            scaler.step(opt)
            scaler.update()
            total_loss += loss.item() * yb.size(0)
            counts += yb.size(0)
        
        ep_train = total_loss / counts
        train_curve.append(ep_train)

        #Validation
        if epoch == 1 or epoch % EVAL_EVERY == 0:
            model.eval()
            val_loss, vcounts = 0.0, 0
            with torch.no_grad():
                for xb, yb in va_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    p = model(xb)
                    val_loss += criterion(p, yb).item() * yb.size(0)
                    vcounts += yb.size(0)
            ep_val = val_loss / vcounts
            val_curve.append(ep_val)
            sch.step(ep_val)

            if ep_val < best_val:
                best_val = ep_val
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                noimp = 0
            else:
                noimp += 1

            # Pruning (Stage 1 only):

            #Stop training immediately if the current model is significantly worse than the best model found in previous trials:
            if stage == 1 and global_best_val < math.inf and MIN_PRUNE_EPOCH and epoch >= MIN_PRUNE_EPOCH:
                
                #Grace Period: Wait until MIN_PRUNE_EPOCH epochs have passed, before allowing pruning. (as some models need some time before they start converging at good performance)
                
                #Early stop if the current model stopped improving compared to its past version: 
                if ep_val > PRUNE_FACTOR * global_best_val:
                    print(f"[{hp['name']}] Pruned at ep {epoch}.")
                    break

            print(f"[{hp['name']}] ep {epoch:03d} | tr {ep_train:.4f} | val {ep_val:.4f}")
            
            if noimp >= PATIENCE:
                print(f"[{hp['name']}] Early stopping.")
                break

    #Save Plot - training curves: 
    pathlib.Path(TRIAL_FIG_DIR).mkdir(parents=True, exist_ok=True)
    try:
        plt.figure(figsize=(7, 4))
        plt.plot(train_curve, label="Train (Scaled)")
        xs = [e for e in range(1, len(train_curve)+1) if e==1 or e%EVAL_EVERY==0][:len(val_curve)]
        plt.plot(xs, val_curve, "o-", label="Val (Scaled)")
        plt.title(f"{hp['name']}")
        plt.legend()
        plt.savefig(os.path.join(TRIAL_FIG_DIR, f"{hp['name']}.png"), dpi=100)
        plt.close()
    except: pass

    #Return Metrics
    #Note: The val_mse returned here is SCALED. 
    #This is fine for picking the best model, but not for the final reporting, as the other models use the MSE from the unscaled data.
    rec = {
        "name": hp["name"],
        "hidden": hp["hidden"],
        "act": hp["act"],
        "dropout": hp["dropout"],
        "batchnorm": hp["batchnorm"],
        "lr": hp["lr"],
        "batch_size": hp["batch_size"],
        "epochs": len(train_curve),
        "val_mse": float(best_val),
        "time": round(time.time() - t0, 1)
    }
    return rec, best_state, best_val


####################################################################


# ------------------------------
# Setup Stage 1 vs 2
# ------------------------------
trials_to_run = []
if STAGE == 1:
    print(f"[INFO] Stage 1: Random Search ({N_TRIALS} trials)")
    for t in range(N_TRIALS):
        trials_to_run.append(sample_hp(t))
else:
    print(f"[INFO] Stage 2: Retraining Top {N_TRIALS} from CSV")
    if not os.path.exists(STAGE1_INPUT_CSV):
        raise FileNotFoundError(f"Missing Stage 1 CSV: {STAGE1_INPUT_CSV}")
    with open(STAGE1_INPUT_CSV, "r") as f:
        reader = csv.DictReader(f)
        rows = sorted(list(reader), key=lambda r: float(r["val_mse"]))[:N_TRIALS]
    
    for i, r in enumerate(rows):
        hp = {
            "name": f"{r['name']}_S2",
            "hidden": ast.literal_eval(r["hidden"]),
            "act": r["act"],
            "dropout": float(r["dropout"]),
            "batchnorm": r["batchnorm"] == "True",
            "lr": float(r["lr"]),
            "batch_size": int(r["batch_size"]),
            "epochs": MAX_EPOCHS
        }
        trials_to_run.append(hp)

####################################################################

# ------------------------------
# Main Loop
# ------------------------------
results = []
best_rec = None
best_state_global = None
GLOBAL_BEST_VAL = math.inf

#CSV Init
with open(RESULTS_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["name","hidden","act","dropout","batchnorm","lr","batch_size","epochs","val_mse","time"])
    w.writeheader()

for i, hp in enumerate(trials_to_run):
    print(f"\n=== Trial {i+1}/{len(trials_to_run)}: {hp['name']} ===")
    rec, state, val_score = train_once(hp, SEED+i, GLOBAL_BEST_VAL, STAGE)
    
    #Save to CSV
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rec.keys())
        w.writerow(rec)
    
    results.append(rec)

    #Track global best: 
    if STAGE == 1:
        GLOBAL_BEST_VAL = min(GLOBAL_BEST_VAL, val_score)

    #Save the best model checkpoint, every time we find a new best: 
    if best_rec is None or val_score < best_rec["val_mse"]:
        best_rec = rec
        best_state_global = state
        
        #Save model + scaler stats
        torch.save({
            "state_dict": state,
            "hp": hp,
            "scaler_mean": scaler_y.mean_, # Save scaler stats to unscale later
            "scaler_scale": scaler_y.scale_,
            "meta": {"job_id": JOB_ID}
        }, BEST_MODEL_PT)
        print(f"[NEW BEST] {hp['name']} (Val MSE Scaled: {val_score:.4f})")

####################################################################

# ------------------------------
# Final Evaluation (Unscaling)
# ------------------------------
print("\n[INFO] Evaluating Best Model on Test Set (UNSCALED)...")
model = FFNN(PCA_DIM, Y.shape[1], best_rec["hidden"], best_rec["act"], best_rec["dropout"], best_rec["batchnorm"]).to(DEVICE)
model.load_state_dict(best_state_global)
model.eval()

te_loader = get_loader(test_ds, best_rec["batch_size"], False)
preds_list, targs_list = [], []

with torch.no_grad():
    for xb, yb in te_loader:
        xb = xb.to(DEVICE)
        p = model(xb)
        preds_list.append(p.cpu().numpy())
        targs_list.append(yb.numpy())

P_scaled = np.vstack(preds_list)
T_scaled = np.vstack(targs_list)

#Inverse scaling transformation back to the log1p space: 
P_log1p = scaler_y.inverse_transform(P_scaled)
T_log1p = scaler_y.inverse_transform(T_scaled)

final_mse = ((P_log1p - T_log1p)**2).mean()

#GPU-friendly Pearson correlation calculation:
yt = torch.tensor(T_log1p, device=DEVICE)
yp = torch.tensor(P_log1p, device=DEVICE)
yt = yt - yt.mean(0, keepdim=True)
yp = yp - yp.mean(0, keepdim=True)
num = (yt * yp).sum(0)
den = torch.sqrt((yt**2).sum(0)) * torch.sqrt((yp**2).sum(0)) + 1e-8
pearson = (num / den).mean().item()

print(f"\n[FINAL RESULTS] Job: {JOB_ID}")
print(f"Best Config: {best_rec['name']}")
print(f"Test MSE (Scaled):   {((P_scaled - T_scaled)**2).mean():.4f}")
print(f"Test MSE (Unscaled) - Comparable to normal FFNN Model: {final_mse:.4f}")
print(f"Test Pearson:        {pearson:.4f}")

# Save Summary
# [FIX] Cast numpy/torch floats to standard python floats for JSON serialization
summary = {
    "stage": STAGE,
    "job_id": JOB_ID,
    "best_config": best_rec,
    "metrics": {
        "test_mse_unscaled": float(final_mse), #The float() is important to prevent an error when writing the JSON file!
        "test_pearson": float(pearson)         #Same as above. 
    }
}
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=4)

print("[INFO] Done.")